# ISL Sign Recognition — Improved Training for 81%+ Accuracy (263 classes)

**Fixes over previous version:**
- Fixed truncation bug in dataset.py (sequences >200 frames were causing shape errors)
- Removed mixup (conflicts with label smoothing, hurts early convergence)
- Disabled label smoothing for first 10 epochs (lets model learn faster initially)
- AdamW + warmup scheduler
- Gradient clipping
- Full visualizations after training

**Branch:** `cnn_new` | **Time:** ~60 min on Colab GPU

In [ ]:
# ── Step 1: Clone cnn_new branch ─────────────────────────────────────
!git clone --branch cnn_new --single-branch https://github.com/BishalDubey27/Major_Project.git
%cd Major_Project
!ls

In [ ]:
# ── Step 2: Install dependencies ─────────────────────────────────────
!pip install mediapipe==0.10.31 transformers==4.44.0 timm joblib tqdm scikit-learn -q
print('Done')

In [ ]:
# ── Step 3: Load keypoints from Google Drive ──────────────────────────
from google.colab import drive
import shutil, os

drive.mount('/content/drive')

# ── Change these paths to match your Drive folder locations ──────────
DRIVE_TRAIN = '/content/drive/MyDrive/include_train_keypoints'
DRIVE_VAL   = '/content/drive/MyDrive/include_val_keypoints'
DRIVE_TEST  = '/content/drive/MyDrive/include_test_keypoints'

os.makedirs('/content/keypoint', exist_ok=True)
shutil.copytree(DRIVE_TRAIN, '/content/keypoint/include_train_keypoints')
shutil.copytree(DRIVE_VAL,   '/content/keypoint/include_val_keypoints')
shutil.copytree(DRIVE_TEST,  '/content/keypoint/include_test_keypoints')

for split in ['include_train_keypoints', 'include_val_keypoints', 'include_test_keypoints']:
    n = len([f for f in os.listdir(f'/content/keypoint/{split}') if f.endswith('.json')])
    print(f'{split}: {n} files')

In [ ]:
# ── Step 4: Fix dataset truncation bug ───────────────────────────────
# The original dataset.py pads but never truncates sequences >200 frames
# This causes shape mismatches. Fix it:
dataset_file = '/content/Major_Project/INCLUDE/dataset.py'
with open(dataset_file) as f:
    content = f.read()

old = '''        final_data = np.concatenate((pose, h1, h2), -1)
        final_data = np.pad(
            final_data,
            ((0, self.max_frame_len - final_data.shape[0]), (0, 0)),
            "constant",
        )'''

new = '''        final_data = np.concatenate((pose, h1, h2), -1)
        if final_data.shape[0] > self.max_frame_len:
            final_data = final_data[:self.max_frame_len]
        else:
            final_data = np.pad(
                final_data,
                ((0, self.max_frame_len - final_data.shape[0]), (0, 0)),
                "constant",
            )'''

if old in content:
    content = content.replace(old, new)
    with open(dataset_file, 'w') as f:
        f.write(content)
    print('dataset.py truncation bug fixed')
else:
    print('Already fixed or pattern not found - checking...')
    print('truncation' in content or 'max_frame_len' in content)

In [ ]:
# ── Step 5: Training ──────────────────────────────────────────────────
import os, sys, json, math
import torch
import torch.nn.functional as F
from torch.utils import data as torch_data
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import numpy as np

sys.path.insert(0, '/content/Major_Project/INCLUDE')
sys.path.insert(0, '/content/Major_Project')

from models.transformer import Transformer
from configs import TransformerConfig
from dataset import KeypointsDataset
from utils import seed_everything, AverageMeter, EarlyStopping, load_json

os.chdir('/content/Major_Project/INCLUDE')

# ── Config ────────────────────────────────────────────────────────────
DATASET       = 'include'       # 263 classes
DATA_DIR      = '/content/keypoint'
SAVE_PATH     = '/content'
EPOCHS        = 80
BATCH_SIZE    = 128
LR            = 3e-4            # Higher LR for faster initial convergence
WEIGHT_DECAY  = 1e-2
WARMUP_EPOCHS = 3               # Short warmup
PATIENCE      = 15
GRAD_CLIP     = 1.0
SEED          = 42
SIZE          = 'small'
LABEL_SMOOTH_START_EPOCH = 10   # No label smoothing for first 10 epochs

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
seed_everything(SEED)

label_map    = load_json(f'label_maps/label_map_{DATASET}.json')
n_classes    = len(label_map)
idx_to_label = {v: k for k, v in label_map.items()}
print(f'Classes: {n_classes}')

config = TransformerConfig(size=SIZE)
model  = Transformer(config=config, n_classes=n_classes).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

train_ds = KeypointsDataset(os.path.join(DATA_DIR, f'{DATASET}_train_keypoints'), use_augs=True,  label_map=label_map, mode='train')
val_ds   = KeypointsDataset(os.path.join(DATA_DIR, f'{DATASET}_val_keypoints'),   use_augs=False, label_map=label_map, mode='val')
train_loader = torch_data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = torch_data.DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(EPOCHS - WARMUP_EPOCHS, 1)
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler  = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
save_file  = os.path.join(SAVE_PATH, f'include_no_cnn_transformer_{SIZE}_improved.pth')
early_stop = EarlyStopping(patience=PATIENCE, mode='max')
best_val   = 0
history    = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'lr': []}

for epoch in range(EPOCHS):
    # Use label smoothing only after warmup phase
    smooth = 0.1 if epoch >= LABEL_SMOOTH_START_EPOCH else 0.0

    # Train
    model.train()
    tl = AverageMeter(); ta = AverageMeter()
    for batch in tqdm(train_loader, desc=f'Ep {epoch+1}/{EPOCHS} train'):
        x, y = batch['data'].to(device), batch['label'].to(device)
        optimizer.zero_grad()
        preds = model(x)
        loss  = F.cross_entropy(preds, y, label_smoothing=smooth)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        tl.update(loss.item())
        ta.update(accuracy_score(y.cpu().numpy(), preds.detach().cpu().argmax(-1).numpy()))
    scheduler.step()

    # Validate
    model.eval()
    vl = AverageMeter(); va = AverageMeter()
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f'Ep {epoch+1}/{EPOCHS} val'):
            x, y = batch['data'].to(device), batch['label'].to(device)
            preds = model(x)
            vl.update(F.cross_entropy(preds, y).item())
            va.update(accuracy_score(y.cpu().numpy(), preds.cpu().argmax(-1).numpy()))

    history['train_loss'].append(tl.avg); history['val_loss'].append(vl.avg)
    history['train_acc'].append(ta.avg);  history['val_acc'].append(va.avg)
    history['lr'].append(scheduler.get_last_lr()[0])

    print(f'Ep {epoch+1}: train_acc={ta.avg:.4f} val_acc={va.avg:.4f} lr={scheduler.get_last_lr()[0]:.6f} smooth={smooth}')

    if va.avg > best_val:
        best_val = va.avg
        torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                    'scheduler': scheduler.state_dict(), 'score': best_val}, save_file)
        print(f'  Saved best: {best_val:.4f}')

    early_stop(save_file, va.avg, model, optimizer, scheduler)
    if early_stop.early_stop:
        print('Early stopping'); break

print(f'Best val accuracy: {round(best_val*100,2)}%')

In [ ]:
# ── Step 6: Training curves ───────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

epochs_ran = range(1, len(history['train_acc']) + 1)
fig = plt.figure(figsize=(18, 5))
gs  = gridspec.GridSpec(1, 3, figure=fig)

ax1 = fig.add_subplot(gs[0])
ax1.plot(epochs_ran, history['train_acc'], label='Train', color='steelblue')
ax1.plot(epochs_ran, history['val_acc'],   label='Val',   color='orange')
ax1.axhline(best_val, color='green', linestyle='--', label=f'Best {best_val:.3f}')
ax1.set_title('Accuracy'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2 = fig.add_subplot(gs[1])
ax2.plot(epochs_ran, history['train_loss'], label='Train', color='steelblue')
ax2.plot(epochs_ran, history['val_loss'],   label='Val',   color='orange')
ax2.set_title('Loss'); ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(True, alpha=0.3)

ax3 = fig.add_subplot(gs[2])
ax3.plot(epochs_ran, history['lr'], color='purple')
ax3.set_title('Learning Rate'); ax3.set_xlabel('Epoch'); ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Step 7: Full evaluation ───────────────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score
import seaborn as sns
from collections import Counter

cp = torch.load(save_file, map_location='cpu', weights_only=False)
model.load_state_dict(cp['model'])
model.eval()

test_ds = KeypointsDataset(os.path.join(DATA_DIR, f'{DATASET}_test_keypoints'), use_augs=False, label_map=label_map, mode='test')
test_loader = torch_data.DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

all_preds = []; all_labels = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc='Testing'):
        x, y = batch['data'].to(device), batch['label']
        all_preds.extend(model(x).cpu().argmax(-1).numpy())
        all_labels.extend(y.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

test_acc    = (all_preds == all_labels).mean()
f1_macro    = f1_score(all_labels, all_preds, average='macro',    zero_division=0)
f1_weighted = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
precision   = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
recall      = recall_score(all_labels, all_preds, average='weighted', zero_division=0)

print('=' * 50)
print(f'Test Accuracy  : {test_acc*100:.2f}%')
print(f'F1 (macro)     : {f1_macro:.4f}')
print(f'F1 (weighted)  : {f1_weighted:.4f}')
print(f'Precision (w)  : {precision:.4f}')
print(f'Recall (w)     : {recall:.4f}')
print('=' * 50)

In [ ]:
# ── Step 8: Pie chart + metrics bar ──────────────────────────────────
correct   = int((all_preds == all_labels).sum())
incorrect = len(all_labels) - correct

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].pie([correct, incorrect],
    labels=[f'Correct\n{correct} ({test_acc*100:.1f}%)', f'Incorrect\n{incorrect} ({(1-test_acc)*100:.1f}%)'],
    colors=['#2ecc71', '#e74c3c'], autopct='%1.1f%%', startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('Correct vs Incorrect Predictions', fontweight='bold')

metrics = ['Accuracy', 'F1 Macro', 'F1 Weighted', 'Precision', 'Recall']
values  = [test_acc, f1_macro, f1_weighted, precision, recall]
bars = axes[1].bar(metrics, values, color=['#3498db','#9b59b6','#e67e22','#1abc9c','#e74c3c'], edgecolor='white')
for bar, val in zip(bars, values):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{val:.3f}', ha='center', fontweight='bold')
axes[1].set_ylim(0, 1.1); axes[1].set_title('Evaluation Metrics', fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/content/metrics_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Step 9: Confusion matrix (top 30 classes) ────────────────────────
top_n = 30
top_ids   = [c for c, _ in Counter(all_labels.tolist()).most_common(top_n)]
top_names = [idx_to_label[i] for i in top_ids]
mask = np.isin(all_labels, top_ids)
fl, fp = all_labels[mask], all_preds[mask]
remap = {old: new for new, old in enumerate(top_ids)}
fl = np.array([remap[l] for l in fl])
fp = np.array([remap[p] if p in remap else -1 for p in fp])
valid = fp >= 0; fl, fp = fl[valid], fp[valid]
cm = confusion_matrix(fl, fp, labels=list(range(top_n)))
cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-8)

fig, ax = plt.subplots(figsize=(20, 16))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
    xticklabels=top_names, yticklabels=top_names, linewidths=0.5, ax=ax, annot_kws={'size': 7})
ax.set_title(f'Confusion Matrix — Top {top_n} Classes (Normalized)', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.xticks(rotation=45, ha='right', fontsize=8); plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Step 10: Per-class F1 (top 20 and bottom 20) ─────────────────────
report = classification_report(all_labels, all_preds, output_dict=True, zero_division=0)
class_f1 = {idx_to_label[int(k)]: v['f1-score'] for k, v in report.items() if k.isdigit()}
sorted_f1 = sorted(class_f1.items(), key=lambda x: x[1], reverse=True)
top20 = sorted_f1[:20]; bottom20 = sorted_f1[-20:]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, data, title, color in [
    (axes[0], top20,    'Top 20 Classes by F1',    '#2ecc71'),
    (axes[1], bottom20, 'Bottom 20 Classes by F1', '#e74c3c')]:
    lp = [d[0] for d in data]; vp = [d[1] for d in data]
    bars = ax.barh(lp, vp, color=color, edgecolor='white')
    for bar, val in zip(bars, vp):
        ax.text(val+0.01, bar.get_y()+bar.get_height()/2, f'{val:.2f}', va='center', fontsize=8)
    ax.set_xlim(0, 1.15); ax.set_title(title, fontweight='bold')
    ax.set_xlabel('F1 Score'); ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('/content/per_class_f1.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Step 11: Download model and all plots ─────────────────────────────
from google.colab import files
for f in [save_file, '/content/training_curves.png', '/content/metrics_summary.png',
          '/content/confusion_matrix.png', '/content/per_class_f1.png']:
    files.download(f)
    print('Downloaded:', f)
print()
print('Place .pth in Major_Project/INCLUDE/ and rename to: include_no_cnn_transformer_small.pth')